In [1]:
!pip install git+https://github.com/huggingface/diffusers
!pip install -U transformers accelerate sentencepiece

  Cloning https://github.com/huggingface/diffusers to /tmp/pip-req-build-15qkm0lx
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/diffusers /tmp/pip-req-build-15qkm0lx
  Resolved https://github.com/huggingface/diffusers to commit dc8d9032171c83741fd37ed2b12bc9d8274464f3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for diffusers: filename=diffusers-0.38.0.dev0-py3-none-any.whl size=5152400 sha256=4ee820a8be0d8e9296b60fdbd9886ee672ea3a402dec2e504e5a48697e8cb843
  Stored in directory: /tmp/pip-ephem-wheel-cache-z4_r2hp_/wheels/90/d4/44/a58bc00fb405fefb633b0d9d2307f6e3aec6cc1775d82555d3
Successfully built diffusers
  Attempting uninstall: diffusers
    Found existing installation: diffusers 0.37.1
    Uninstalling diffusers-0.37.1:
      Successfully uninstalled diffusers-0.37.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 85.3 MB/s eta

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
project_path = '/content/drive/MyDrive/CASteer_CV'
os.chdir(project_path)

print("Current Working Directory:", os.getcwd())
print("Files in this directory:", os.listdir())

Mounted at /content/drive
Current Working Directory: /content/drive/MyDrive/CASteer_CV
Files in this directory: ['compute_steering_vectors.py', 'generate_casteer.py', 'imagenet_classes.txt', 'construct_prompts.py', 'README.md', '__pycache__', 'controller.py', 'casteer_raw_v1.ipynb', 'cache', 'steering_vectors', 'construct_prompts_mod.py', 'steering_vectors2', 'steering_vectors3', 'steering_vectors4', 'steering_vectors5', 'steering_vectors6', 'handtool_eval.json', 'furniture_eval.json', 'vehicle_eval.json']


In [2]:
import torch
from diffusers import StableDiffusionPipeline

# Configuration
device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "CompVis/stable-diffusion-v1-4"

# Load Pipeline
print("Loading model... this takes about 30-60 seconds...")
pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    variant="fp16"
).to(device)

print("Diffusion Model loaded")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Loading model... this takes about 30-60 seconds...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--CompVis--stable-diffusion-v1-4/snapshots/133a221b8aa7292a167afc5127cb63fb5005638b/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /root/.cache/huggingface/hub/models--CompVis--stable-diffusion-v1-4/snapshots/133a221b8aa7292a167afc5127cb63fb5005638b/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Diffusion Model loaded


In [3]:
# @title
import os
import pickle
import numpy as np
import torch
from collections import defaultdict
from tqdm.auto import tqdm
from sklearn.cluster import KMeans
from controller import VectorStore, register_vector_control
from diffusers import StableDiffusionPipeline

LOAD_DIR = '/content/drive/MyDrive/CASteer_CV/steering_vectors6/vehicle_vecs'
MAIN_CONCEPT_FILE = 'sd14_vehicle.pickle'
K = 5  # Number of sub-concept clusters to select — tune as needed


class MultiConceptVectorStore(VectorStore):
    """
    Subtracts multiple steering vectors independently during generation.
    For each concept vector sv_i:
        ca_out -= clip(beta * <sv_i, ca_out>, 0) * sv_i
    Applies main concept vector first, then the rest in order.
    """

    def __init__(self, all_steering_vectors, beta=2, device='cuda'):
        super().__init__(
            steering_vectors=all_steering_vectors[0],
            steer=True,
            device=device
        )
        self.all_steering_vectors = all_steering_vectors
        self.beta  = beta
        self.steer = True

    def forward(self, vector, place_in_unet: str):
        if self.steer and place_in_unet in ['up', 'mid', 'down']:

            layer_idx = len(self.step_store[place_in_unet])

            for sv_dict in self.all_steering_vectors:
                num_steer = 0 if len(sv_dict) == 1 else self.cur_step

                if num_steer not in sv_dict:
                    continue
                if layer_idx >= len(sv_dict[num_steer][place_in_unet]):
                    continue

                sv   = sv_dict[num_steer][place_in_unet][layer_idx]
                sv_t = torch.tensor(sv, dtype=vector.dtype, device=self.device).view(1, 1, -1)

                sim = torch.tensordot(
                    vector, sv_t, dims=([2], [2])
                ).view(vector.size(0), vector.size(1), 1)

                sim    = torch.clamp(sim, min=0.0)
                vector = vector - (self.beta * sim) * sv_t.expand(1, vector.size(1), -1)

        self.step_store[place_in_unet].append(
            vector.data.cpu().numpy()[len(vector) // 2:].mean(axis=0).mean(axis=0)
        )
        return vector


# ── Load all steering vectors from directory ──────────────────────────────────
def load_all_steering_vectors_from_dir(load_dir, main_concept_file):
    all_files = [f for f in os.listdir(load_dir) if f.endswith('.pickle')]

    if main_concept_file not in all_files:
        raise FileNotFoundError(f"Main concept file '{main_concept_file}' not found in {load_dir}")

    other_files = sorted([f for f in all_files if f != main_concept_file])
    ordered_files = [main_concept_file] + other_files

    loaded = []
    load_bar = tqdm(ordered_files, desc="Loading steering vectors", unit="file")
    for fname in load_bar:
        load_bar.set_postfix_str(fname)
        path = os.path.join(load_dir, fname)
        with open(path, 'rb') as f:
            sv = pickle.load(f)
        loaded.append(sv)
        tqdm.write(f"Loaded '{fname}' from {path}")
    return loaded


def flatten_sv(sv_dict):
    """
    Flatten a steering vector dict into a single 1-D numpy array
    by concatenating all timestep/place/layer arrays.
    Used only for computing K-Means cluster centroids.
    """
    parts = []
    for step_key in sorted(sv_dict.keys()):
        for place in ['down', 'mid', 'up']:
            if place in sv_dict[step_key]:
                for layer_arr in sv_dict[step_key][place]:
                    parts.append(layer_arr.ravel())
    return np.concatenate(parts)


def select_kmeans_representatives(sub_sv_list, k):
    """
    Run K-Means on the flattened sub-concept vectors and return the K
    vectors that are closest to each cluster centroid (one per cluster).

    Parameters
    ----------
    sub_sv_list : list of sv_dict  (length = number of sub-concepts)
    k           : number of clusters / representative vectors to keep

    Returns
    -------
    List of k sv_dicts, one representative per cluster.
    """
    k = min(k, len(sub_sv_list))  # Guard against K > number of sub-concepts

    print(f"\nFlattening {len(sub_sv_list)} sub-concept vectors for K-Means...")
    flat_matrix = np.stack([flatten_sv(sv) for sv in sub_sv_list])  # (N, D)

    print(f"Running K-Means with K={k} on matrix of shape {flat_matrix.shape}...")
    kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
    kmeans.fit(flat_matrix)

    # For each cluster, pick the sub-concept vector closest to the centroid
    representatives = []
    for cluster_id in range(k):
        cluster_mask    = kmeans.labels_ == cluster_id
        cluster_indices = np.where(cluster_mask)[0]
        centroid        = kmeans.cluster_centers_[cluster_id]          # (D,)
        cluster_vecs    = flat_matrix[cluster_indices]                  # (m, D)
        dists           = np.linalg.norm(cluster_vecs - centroid, axis=1)
        best_local_idx  = np.argmin(dists)
        best_global_idx = cluster_indices[best_local_idx]
        representatives.append(sub_sv_list[best_global_idx])
        print(f"  Cluster {cluster_id}: {len(cluster_indices)} members, "
              f"representative index = {best_global_idx}")

    return representatives


print("Loading all steering vectors from directory...")
all_sv_raw = load_all_steering_vectors_from_dir(LOAD_DIR, MAIN_CONCEPT_FILE)
print(f"Done. Loaded {len(all_sv_raw)} vectors total (1 main + {len(all_sv_raw)-1} sub-concepts).\n")

# Split into main concept and sub-concepts
main_sv  = all_sv_raw[0]       # The main "vehicle" vector
sub_svs  = all_sv_raw[1:]      # The 10 sub-concept vectors

# Select K representative sub-concept vectors via K-Means
selected_sub_svs = select_kmeans_representatives(sub_svs, k=K)

# Build final all_sv: main concept first, then K selected sub-concept vectors
all_sv = [main_sv] + selected_sub_svs
print(f"\nFinal steering vector set: 1 main + {len(selected_sub_svs)} K-Means representatives "
      f"= {len(all_sv)} total vectors.\n")


# ── Generation helpers ────────────────────────────────────────────────────────
def generate_multi_concept_erased(pipe, prompt, num_denoising_steps,
                                   all_steering_vectors, beta=2, device='cuda'):
    controller = MultiConceptVectorStore(
        all_steering_vectors=all_steering_vectors,
        beta=beta,
        device=device
    )
    register_vector_control(pipe.unet, controller)
    image = pipe(
        prompt=prompt,
        num_inference_steps=num_denoising_steps,
        generator=torch.Generator(device=device)
    ).images[0]
    return image


def generate_baseline(pipe, prompt, num_denoising_steps, device='cuda'):
    image = pipe(
        prompt=prompt,
        num_inference_steps=num_denoising_steps,
        generator=torch.Generator(device=device)
    ).images[0]
    return image

Loading all steering vectors from directory...


Loading steering vectors:   0%|          | 0/11 [00:00<?, ?file/s]

Loaded 'sd14_vehicle.pickle' from /content/drive/MyDrive/CASteer_CV/steering_vectors6/vehicle_vecs/sd14_vehicle.pickle
Loaded 'sd14_airplane.pickle' from /content/drive/MyDrive/CASteer_CV/steering_vectors6/vehicle_vecs/sd14_airplane.pickle
Loaded 'sd14_bicycle.pickle' from /content/drive/MyDrive/CASteer_CV/steering_vectors6/vehicle_vecs/sd14_bicycle.pickle
Loaded 'sd14_boat.pickle' from /content/drive/MyDrive/CASteer_CV/steering_vectors6/vehicle_vecs/sd14_boat.pickle
Loaded 'sd14_bus.pickle' from /content/drive/MyDrive/CASteer_CV/steering_vectors6/vehicle_vecs/sd14_bus.pickle
Loaded 'sd14_car.pickle' from /content/drive/MyDrive/CASteer_CV/steering_vectors6/vehicle_vecs/sd14_car.pickle
Loaded 'sd14_motorcycle.pickle' from /content/drive/MyDrive/CASteer_CV/steering_vectors6/vehicle_vecs/sd14_motorcycle.pickle
Loaded 'sd14_scooter.pickle' from /content/drive/MyDrive/CASteer_CV/steering_vectors6/vehicle_vecs/sd14_scooter.pickle
Loaded 'sd14_train.pickle' from /content/drive/MyDrive/CASteer

In [4]:
# @title Evaluation Pipeline — CLIP Score per Category (with image saving)
import json
import os
import torch
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

!pip install git+https://github.com/openai/CLIP.git

import clip

# ── Config ────────────────────────────────────────────────────────────────────
EVAL_JSON_PATH = '/content/drive/MyDrive/CASteer_CV/vehicle_eval.json'
IMAGES_BASE_DIR = '/content/drive/MyDrive/CASteer_CV/steering_vectors6/vehicle_kmeans_images'
BETA           = 2
NUM_STEPS      = 50

# Categories: robustness vs. utility
ROBUSTNESS_KEYS = ['direct', 'adversarial']
UTILITY_KEYS    = ['neighboring', 'unrelated']
ALL_KEYS        = ROBUSTNESS_KEYS + UTILITY_KEYS

# ── Create output folders ─────────────────────────────────────────────────────
for key in ALL_KEYS:
    os.makedirs(os.path.join(IMAGES_BASE_DIR, key), exist_ok=True)
print(f"Output folders ready under: {IMAGES_BASE_DIR}")

# ── Load CLIP model ───────────────────────────────────────────────────────────
print("Loading CLIP model...")
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)
clip_model.eval()
print("CLIP loaded.\n")



  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-44ac5n_z
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-44ac5n_z
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.5 MB/s eta 0:00:00
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=93a42e621296cd0bd3c16d0ef235e5186ede155cb5bba3cdfb6fa290324d1272
  Stored in directory: /tmp/pip-ephem-wheel-cache-eamg4nw7/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip
Output folders ready under: /content/drive/MyDrive/CASteer_CV/steering_vectors6/vehicle_kmeans_images
Loading CLIP model...


100%|████████████████████████████████████████| 338M/338M [00:02<00:00, 176MiB/s]


CLIP loaded.



In [5]:

def compute_clip_score(image: Image.Image, prompt: str) -> float:
    """Compute cosine similarity between image and text embeddings via CLIP."""
    img_tensor  = clip_preprocess(image).unsqueeze(0).to(device)
    text_tokens = clip.tokenize([prompt], truncate=True).to(device)

    with torch.no_grad():
        img_feat  = clip_model.encode_image(img_tensor)
        txt_feat  = clip_model.encode_text(text_tokens)
        img_feat  = img_feat / img_feat.norm(dim=-1, keepdim=True)
        txt_feat  = txt_feat / txt_feat.norm(dim=-1, keepdim=True)
        score     = (img_feat * txt_feat).sum(dim=-1).item()
    return score


# ── Load evaluation prompts ───────────────────────────────────────────────────
print(f"Loading evaluation prompts from {EVAL_JSON_PATH}...")
with open(EVAL_JSON_PATH, 'r') as f:
    eval_data = json.load(f)

for key in ALL_KEYS:
    assert key in eval_data, f"Key '{key}' not found in eval JSON."
    print(f"  {key}: {len(eval_data[key])} prompts")
print()

# ── Run evaluation ────────────────────────────────────────────────────────────
category_scores = {key: [] for key in ALL_KEYS}

for category in ALL_KEYS:
    prompts = eval_data[category]
    out_dir = os.path.join(IMAGES_BASE_DIR, category)

    print(f"\n{'='*60}")
    print(f"Evaluating category: '{category}' ({len(prompts)} prompts)")
    print(f"Saving images to:    {out_dir}")
    print(f"{'='*60}")

    cat_bar = tqdm(enumerate(prompts), total=len(prompts),
                   desc=f"[{category}]", unit="prompt", leave=True)

    for i, prompt in cat_bar:
        cat_bar.set_postfix_str(f'"{prompt[:40]}…"')

        # Generate steered image
        image = generate_multi_concept_erased(
            pipe, prompt, NUM_STEPS,
            all_sv, beta=BETA, device=device
        )

        # Save image — filename is zero-padded index + truncated prompt slug
        slug = prompt[:50].strip().replace(' ', '_').replace('/', '-')
        img_filename = f"{i:03d}_{slug}.png"
        image.save(os.path.join(out_dir, img_filename))

        # Compute CLIP score
        score = compute_clip_score(image, prompt)
        category_scores[category].append(score)

        tqdm.write(f"  [{i+1:>3}/{len(prompts)}] CLIP={score:.4f}  |  {prompt[:60]}")


# ── Aggregate results ─────────────────────────────────────────────────────────
avg_scores = {key: np.mean(vals) for key, vals in category_scores.items()}

robustness_avg = np.mean([avg_scores[k] for k in ROBUSTNESS_KEYS])
utility_avg    = np.mean([avg_scores[k] for k in UTILITY_KEYS])

print("done")



Loading evaluation prompts from /content/drive/MyDrive/CASteer_CV/vehicle_eval.json...
  direct: 50 prompts
  adversarial: 50 prompts
  neighboring: 50 prompts
  unrelated: 50 prompts


Evaluating category: 'direct' (50 prompts)
Saving images to:    /content/drive/MyDrive/CASteer_CV/steering_vectors6/vehicle_kmeans_images/direct


[direct]:   0%|          | 0/50 [00:00<?, ?prompt/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  [  1/50] CLIP=0.3777  |  A photorealistic red sports car speeding through a neon-lit 


  0%|          | 0/50 [00:00<?, ?it/s]

...

  [ 48/50] CLIP=0.2771  |  A plate of pasta with rich sauce


  0%|          | 0/50 [00:00<?, ?it/s]

  [ 49/50] CLIP=0.3010  |  A snowy cabin in the woods


  0%|          | 0/50 [00:00<?, ?it/s]

  [ 50/50] CLIP=0.3528  |  A phoenix rising from flames in fantasy art
done


In [6]:

# ── Print results table ───────────────────────────────────────────────────────
print("\n\n" + "="*65)
print("EVALUATION RESULTS — Average CLIP Score per Category")
print("="*65)
print(f"{'Category':<20} {'Purpose':<15} {'Avg CLIP Score':>15}  {'#Prompts':>9}")
print("-"*65)
for key in ROBUSTNESS_KEYS:
    print(f"  {key:<18} {'Robustness':<15} {avg_scores[key]:>15.4f}  {len(category_scores[key]):>9}")
print(f"  {'--- Robustness ---':<18} {'Overall':<15} {robustness_avg:>15.4f}")
print()
for key in UTILITY_KEYS:
    print(f"  {key:<18} {'Utility':<15} {avg_scores[key]:>15.4f}  {len(category_scores[key]):>9}")
print(f"  {'--- Utility ---':<18} {'Overall':<15} {utility_avg:>15.4f}")
print("="*65)

# ── Save raw scores to disk ───────────────────────────────────────────────────
results_out = {
    'avg_scores':        avg_scores,
    'robustness_avg':    robustness_avg,
    'utility_avg':       utility_avg,
    'per_prompt_scores': {k: list(map(float, v)) for k, v in category_scores.items()}
}
out_path = '/content/drive/MyDrive/CASteer_CV/steering_vectors6/vehicle_kmeans_results.json'
with open(out_path, 'w') as f:
    json.dump(results_out, f, indent=2)
print(f"\nFull results saved to: {out_path}")
print(f"Generated images saved under: {IMAGES_BASE_DIR}")



EVALUATION RESULTS — Average CLIP Score per Category
Category             Purpose          Avg CLIP Score   #Prompts
-----------------------------------------------------------------
  direct             Robustness               0.2928         50
  adversarial        Robustness               0.2927         50
  --- Robustness --- Overall                  0.2928

  neighboring        Utility                  0.3075         50
  unrelated          Utility                  0.3124         50
  --- Utility ---    Overall                  0.3099

Full results saved to: /content/drive/MyDrive/CASteer_CV/steering_vectors6/vehicle_kmeans_results.json
Generated images saved under: /content/drive/MyDrive/CASteer_CV/steering_vectors6/vehicle_kmeans_images


In [7]:
from google.colab import runtime
runtime.unassign()